# Parametric model of plate with multiple holes

In [14]:
%pip install ansys-mapdl-core
%pip install pyvista
%pip install ansys-mapdl-core[graphics]

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [15]:
import sys
import pyvista as pv
import ansys.mapdl.core as pymapdl

print(sys.version)
print(pv.__version__)
print(pymapdl.__version__)

3.13.12 (tags/v3.13.12:1cbe481, Feb  3 2026, 18:22:25) [MSC v.1944 64 bit (AMD64)]
0.47.0
0.72.1


In [16]:
from ansys.mapdl import core as pymapdl
import pyvista as pv

### Objective:
In this example we'll setup a parametric model in PyMAPDL for a rectangular plate with multiple holes. The model is setup such that one can change the dimensions of the plate, the number of holes and their radius, the material properties and the pressure applied.

The learning objectives of this demo are:
* Launch PyMAPDL on a local machine
* Setup and solve a parametric model using PyMAPDL
* Interactive plotting of CAD, mesh, and results in Pythonic interface.

### Model Details
<div>
    <img src="attachment:plate_with_hole.png" width="500"/>
</div>

#### Model parameters:
* Length, width and depth of the plate
* Number of holes
* Raidus of the holes
* Material properties (Young's modulus and Poisson's ratio)
* Applied pressure

## Step 1 - define all parameters

In [17]:
# All units in (m, Kg, s)
LENGTH = 5
WIDTH = 2.5
DEPTH = 0.1
RADIUS = 0.5
NUM = 3

E = 2e11
NU = 0.27

PRESSURE = 1000

# Step 2 - launch MAPDL and create geometry

In [18]:
from ansys.mapdl.core import launch_mapdl
mapdl = launch_mapdl()
print(mapdl)

ERROR - pymapdl_global -  launcher - launch_mapdl - An error occurred when connecting to MAPDL.


MapdlConnectionError: Unable to connect to MAPDL gRPC instance at dns:///127.0.0.1:50058.
Reached either maximum amount of connection attempts (5) or timeout (45 s). The MAPDL process has died PID: 25952.

In [13]:
from ansys.mapdl.core import launch_mapdl
mapdl = launch_mapdl()

mapdl.clear()
mapdl.prep7()
mapdl.block(0, LENGTH, 0, WIDTH, 0, DEPTH)
for i in range(1,NUM+1):
    mapdl.cyl4(i*LENGTH/(NUM+1),WIDTH/2,RADIUS,'','','',2*DEPTH)
mapdl.vsbv(1,'all')
mapdl.vplot('all')

ModuleNotFoundError: Graphic libraries are required to use this method.
You can install this using `pip install ansys-mapdl-core[graphics]`.

## Step 3 - define material properties, mesh attributes and generate mesh.

In [3]:
mapdl.lesize("ALL", 0.15, layer1=1)

mapdl.mp('ex',1,E)
mapdl.mp('nuxy',1,NU)

mapdl.et(1,'SOLID186')
mapdl.mshape(1, "3D")
mapdl.mshkey(0)
mapdl.vmesh('all')
mapdl.eplot()

ViewInteractiveWidget(height=768, layout=Layout(height='auto', width='100%'), width=1024)

## Step 4 - apply loads and boundary conditions

In [4]:
mapdl.nsel('s','loc','x',0)
mapdl.d('all','all',0)

mapdl.nsel('s','loc','x', LENGTH)
mapdl.sf('all','pres',PRESSURE)

mapdl.allsel()
mapdl.finish()

'***** ROUTINE COMPLETED *****  CP =         4.812'

## Step 4 - solve the static problem

In [5]:
mapdl.slashsolu()
mapdl.solve()
mapdl.finish()

'FINISH SOLUTION PROCESSING\n\n\n ***** ROUTINE COMPLETED *****  CP =         5.812'

In [7]:
# enter the solver routine and solve 
mapdl.slashsolu()
output = mapdl.solve()

print(output)

*****  ANSYS SOLVE    COMMAND  *****

 *** NOTE ***                            CP =       6.094   TIME= 16:27:00
 Present time 0 is less than or equal to the previous time.  Time will   
 default to 3.                                                           

 *** ANSYS - ENGINEERING ANALYSIS SYSTEM  RELEASE 2021 R1          21.1     ***
 DISTRIBUTED ANSYS Mechanical Enterprise                       

 00000000  VERSION=WINDOWS x64   16:27:00  APR 21, 2022 CP=      6.109

                                                                               



                      L O A D   S T E P   O P T I O N S

   LOAD STEP NUMBER. . . . . . . . . . . . . . . .     3
   TIME AT END OF THE LOAD STEP. . . . . . . . . .  3.0000    
   NUMBER OF SUBSTEPS. . . . . . . . . . . . . . .     1
   STEP CHANGE BOUNDARY CONDITIONS . . . . . . . .    NO
   PRINT OUTPUT CONTROLS . . . . . . . . . . . . .NO PRINTOUT
   DATABASE OUTPUT CONTROLS. . . . . . . . . . . .ALL DATA WRITTEN
                  

## Step 5 - plot the stress contours for the model

In [6]:
result = mapdl.result
result.plot_principal_nodal_stress(0,'seqv',background='w',show_edges=True,text_color='k',add_text=True)

ViewInteractiveWidget(height=768, layout=Layout(height='auto', width='100%'), width=1024)

## Step 6 - exit MAPDL

In [7]:
mapdl.exit()